# IoU 与 NMS：几何计算到检测框去重

本笔记推导交并比 IoU，解释贪心 NMS，并给出可直接运行的 NumPy 实现和可视化。

## 1. BBox 与 IoU 推导

使用角点格式 $b=(x_1,y_1,x_2,y_2)$：左上角为 $(x_1,y_1)$，右下角为 $(x_2,y_2)$。面积为

$$A(b)=(x_2-x_1)(y_2-y_1).$$

对两个框 $A,B$，交集框的边界为

$$x_1^I=\max(x_1^A,x_1^B),\quad y_1^I=\max(y_1^A,y_1^B),$$
$$x_2^I=\min(x_2^A,x_2^B),\quad y_2^I=\min(y_2^A,y_2^B).$$

交集宽高必须截断为非负：

$$w_I=\max(0,x_2^I-x_1^I),\quad h_I=\max(0,y_2^I-y_1^I),\quad A_I=w_Ih_I.$$

由容斥原理，$A_U=A(A)+A(B)-A_I$，所以

$$\boxed{\operatorname{IoU}(A,B)=\frac{A_I}{A_U}}.$$

完全重合时为 $1$，无重叠时为 $0$。MOT 常使用 $1-\operatorname{IoU}$ 作为 motion cost。

In [ ]:
import numpy as np


def iou_one_to_one(box_a, box_b):
    # 两个 xyxy 格式框的 IoU
    a = np.asarray(box_a, dtype=float)
    b = np.asarray(box_b, dtype=float)
    left_top = np.maximum(a[:2], b[:2])
    right_bottom = np.minimum(a[2:], b[2:])
    inter_wh = np.maximum(0.0, right_bottom - left_top)
    inter_area = np.prod(inter_wh)
    area_a = np.prod(np.maximum(0.0, a[2:] - a[:2]))
    area_b = np.prod(np.maximum(0.0, b[2:] - b[:2]))
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0

A, B = [10, 10, 50, 50], [30, 20, 70, 60]
print(f'IoU(A, B) = {iou_one_to_one(A, B):.4f}')

## 2. 矩阵化 IoU

检测与跟踪需要同时比较多组框。对 $N$ 个框和 $M$ 个框，构造矩阵

$$I\in\mathbb{R}^{N\times M},\qquad I_{ij}=\operatorname{IoU}(A_i,B_j).$$

以下用 NumPy broadcasting 避免双重 Python 循环。

In [ ]:
def pairwise_iou(boxes_a, boxes_b):
    # 返回 (N, M) 的 xyxy IoU 矩阵
    a = np.asarray(boxes_a, dtype=float)[:, None, :]
    b = np.asarray(boxes_b, dtype=float)[None, :, :]
    left_top = np.maximum(a[..., :2], b[..., :2])
    right_bottom = np.minimum(a[..., 2:], b[..., 2:])
    inter = np.prod(np.maximum(0.0, right_bottom - left_top), axis=-1)
    area_a = np.prod(np.maximum(0.0, a[..., 2:] - a[..., :2]), axis=-1)
    area_b = np.prod(np.maximum(0.0, b[..., 2:] - b[..., :2]), axis=-1)
    union = area_a + area_b - inter
    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)

boxes_a = np.array([[10, 10, 50, 50], [60, 10, 90, 50]])
boxes_b = np.array([[30, 20, 70, 60], [62, 12, 88, 48], [100, 100, 120, 120]])
print(pairwise_iou(boxes_a, boxes_b))

## 3. Greedy NMS

检测器对同一对象常产生多个重叠候选框。给定候选 $(b_i,s_i)$ 和阈值 $\tau$，NMS：

1. 按 score 从高到低排序；
2. 保留最高分框 $b^*$；
3. 删除满足 $\operatorname{IoU}(b^*,b_i)>\tau$ 的其余框；
4. 重复至候选为空。

这是贪心后处理，复杂度通常为 $O(N^2)$。

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    # 返回保留框在原数组中的下标
    boxes = np.asarray(boxes, dtype=float)
    scores = np.asarray(scores, dtype=float)
    if boxes.ndim != 2 or boxes.shape[1] != 4 or len(boxes) != len(scores):
        raise ValueError('boxes 必须为 (N, 4)，scores 必须为 (N,)')
    order, keep = scores.argsort()[::-1], []
    while order.size:
        current = order[0]
        keep.append(current)
        if order.size == 1:
            break
        remaining = order[1:]
        ious = pairwise_iou(boxes[current:current + 1], boxes[remaining])[0]
        order = remaining[ious <= iou_threshold]
    return np.array(keep, dtype=int)

boxes = np.array([[10, 10, 50, 50], [12, 12, 52, 52], [15, 10, 55, 50],
                  [70, 15, 105, 55], [72, 17, 107, 57]])
scores = np.array([.95, .80, .70, .92, .60])
keep = nms(boxes, scores, .5)
print('kept indices:', keep)
print('kept boxes:', boxes[keep])

## 4. 可视化与局限

绿色框是保留结果，红色虚线框是被抑制的候选。降低阈值会更激进地删除框。在拥挤场景，两个不同对象也可能有高 IoU，标准 NMS 会造成漏检；Soft-NMS 选择降低分数而不直接删除。

检测 NMS 在单帧删除重复 detection；Track NMS 则删除重复轨迹。DETR/MOTR 的 set prediction 试图从训练阶段减少重复预测。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(7, 4))
ax.set_xlim(0, 125); ax.set_ylim(75, 0); ax.set_aspect('equal')
for i, (box, score) in enumerate(zip(boxes, scores)):
    x1, y1, x2, y2 = box
    kept = i in set(keep)
    color, style = ('tab:green', '-') if kept else ('tab:red', '--')
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                           edgecolor=color, linestyle=style, linewidth=2))
    ax.text(x1, y1-2, f'{i}: {score:.2f}', color=color)
ax.set_title('NMS: green kept, red suppressed')
ax.set_xlabel('x'); ax.set_ylabel('y'); plt.show()